In [ ]:
# ==============================================================================
# 1. COLAB / NOTEBOOK INITIAL SETUP (Dependencies & Models)
# ==============================================================================
import os
import torch

# Force CUDA check
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found! Go to Runtime > Change runtime type > T4 GPU")
print(f"GPU Detected: {torch.cuda.get_device_name(0)}")

# Install Standard Libraries
!pip install -q supervision laptrack pandas

# Install SAM 2 from Source
if not os.path.exists("sam2"):
    print("Cloning SAM 2 repository...")
    !git clone https://github.com/facebookresearch/sam2.git
    !cd sam2 && pip install -e .

# Download Model Checkpoints
if not os.path.exists("sam2/checkpoints/sam2.1_hiera_small.pt"):
    print("Downloading SAM 2 checkpoints...")
    !cd sam2/checkpoints && ./download_ckpts.sh
else:
    print("Checkpoints already present.")

print("Setup Complete! 'Small' model is ready.")


In [ ]:
# ==============================================================================
# 2. STANDARD IMPORTS & ENVIRONMENT SETUP
# ==============================================================================
import sys
import json
import time
import argparse
from typing import List, Dict

import numpy as np
import cv2

# Add the SAM2 directory to sys.path
sys.path.append('/content/sam2')

# Jupyter/Colab arg fix
if 'google.colab' in sys.modules or 'ipykernel' in sys.modules:
    sys.argv = [sys.argv[0]]

def import_sam2():
    try:
        from sam2.build_sam import build_sam2
        from sam2.sam2_image_predictor import SAM2ImagePredictor
        return build_sam2, SAM2ImagePredictor
    except:
        pass

    for path in ["/content/sam2", "./sam2", "../sam2"]:
        if os.path.isdir(path):
            sys.path.insert(0, path)
            for m in list(sys.modules):
                if m.startswith("sam2"):
                    del sys.modules[m]
            try:
                from sam2.build_sam import build_sam2
                from sam2.sam2_image_predictor import SAM2ImagePredictor
                return build_sam2, SAM2ImagePredictor
            except:
                pass

    raise ImportError("sam2 import failed")

In [10]:
# ==============================================================================
# 3. GLOBAL CONFIGURATIONS & PARAMETERS
# ==============================================================================
# --- SAM 2 Detection Params ---
MIN_AREA = 80
MAX_AREA = 50000
BACKGROUND_MEDIAN_FRAMES = 25
RESIZE_SCALE = 1.0
OUT_MASKS = "masks"                # directory to store per-detection mask PNGs
IN_VIDEO = "r3-lowflow-normal-input.mp4"      # input video path
OUT_JSON = "detections_with_masks.json"
OUT_OVERLAY = "detection_overlay.mp4"  # optional overlay video of detections (set to None to skip)
WRITE_OVERLAY = True

# FLUX SETTINGS (THE FIX)
FLUX_THRESHOLD = 15
MAX_FRAMES_TO_PROCESS = 50

CHECKPOINT = "./sam2/checkpoints/sam2.1_hiera_small.pt"
CONFIG_FILE = "configs/sam2.1/sam2.1_hiera_s.yaml"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Initializing SAM 2 SMALL on {device}...")

# --- Mask-IoU Tracker Params ---
TRACKER_CONFIG = {
    "input_video": OUT_OVERLAY,      # This gets overridden by CLI args in main()
    "input_json": OUT_JSON,                # Linked to SAM 2 output
    "input_masks": OUT_MASKS,                 # Linked to SAM 2 output
    "output_json": "tracks_mask_iou_fix1.json",
    "output_video": "out_mask_iou_fix1.mp4",

    # --- association ---
    "iou_match_thresh": 0.10,        # accept track ↔ detection
    "iou_spawn_thresh": 0.05,        # below this → new track
    "max_centroid_dist": 120,        # gating only (pixels)

    # --- persistence ---
    "max_missed_frames": 8,          # occlusion tolerance
    "min_confirm_frames": 3,         # suppress noise tracks

    # --- visualization ---
    "draw_trails": True,
    "trail_length": 20,
    "mask_alpha": 0.5
}

Initializing SAM 2 SMALL on cuda...


In [11]:

# ==============================================================================
# 4. SAM 2 HELPER FUNCTIONS
# ==============================================================================
def ensure_dir(p):
    if not os.path.exists(p):
        os.makedirs(p)

def mask_to_polygon(mask, epsilon_frac=0.01):
    mask_u8 = (mask.astype('uint8') * 255).copy()
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    c = max(contours, key=cv2.contourArea)
    peri = cv2.arcLength(c, True)
    eps = max(1.0, epsilon_frac * peri)
    approx = cv2.approxPolyDP(c, eps, True)
    pts = [[int(p[0][0]), int(p[0][1])] for p in approx]
    return pts

def mask_to_bbox(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return [0,0,0,0]
    x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
    return [x1, y1, x2-x1+1, y2-y1+1]


# ==============================================================================
# 5. CORE LOGIC: CANDIDATE GENERATION & BACKGROUND ESTIMATION
# ==============================================================================
def estimate_median_background(video_path, num_frames=BACKGROUND_MEDIAN_FRAMES, scale=1.0):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    indices = np.linspace(0, max(0,total-1), min(num_frames, max(1,total)), dtype=int)
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, f = cap.read()
        if not ret:
            continue
        if scale != 1.0:
            f = cv2.resize(f, (int(f.shape[1]*scale), int(f.shape[0]*scale)))
        frames.append(f.astype(np.uint8))
    cap.release()
    if not frames:
        return None
    med = np.median(np.stack(frames, axis=0).astype(np.uint8), axis=0).astype(np.uint8)
    return med

def generate_candidate_boxes(frame_bgr, prev_frame, next_frame, bg_median):
    # 1. Background Subtraction (Standard)
    diff = cv2.absdiff(frame_bgr, bg_median)
    gray_diff = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, bg_mask = cv2.threshold(gray_diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 2. Temporal Flux (Motion Check)
    if prev_frame is not None and next_frame is not None:
        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        gray_prev = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        gray_next = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)

        d1 = cv2.absdiff(gray, gray_prev)
        d2 = cv2.absdiff(gray, gray_next)

        _, t1 = cv2.threshold(d1, FLUX_THRESHOLD, 255, cv2.THRESH_BINARY)
        _, t2 = cv2.threshold(d2, FLUX_THRESHOLD, 255, cv2.THRESH_BINARY)

        flux_mask = cv2.bitwise_and(t1, t2)
        flux_mask = cv2.morphologyEx(flux_mask, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))

        # 3. INTERSECTION
        final_mask = cv2.bitwise_and(bg_mask, flux_mask)
    else:
        final_mask = bg_mask

    # Cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_CLOSE, kernel, iterations=1)

    contours, _ = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < MIN_AREA or area > MAX_AREA:
            continue
        x,y,w,h = cv2.boundingRect(c)
        boxes.append([int(x), int(y), int(x+w), int(y+h)])
    return boxes


# ==============================================================================
# 6. SAM2 WRAPPER CLASS
# ==============================================================================
class SAMWrapper:
    def __init__(self, config, checkpoint, device):
        self.predictor = None
        try:
            from sam2.build_sam import build_sam2
            from sam2.sam2_image_predictor import SAM2ImagePredictor
        except Exception as e:
            print("[SAMWrapper] Could not import SAM2 modules:", e)
            raise

        curr_dir = os.getcwd()
        try:
            try:
                print("[SAMWrapper] Building SAM2 model...")
                sam2_model = build_sam2(config, checkpoint, device=device)
            except FileNotFoundError:
                print("❌ Error: Checkpoint or Config not found. Check paths.")
                os.chdir(curr_dir)
                raise
            finally:
                os.chdir(curr_dir)
            self.predictor = SAM2ImagePredictor(sam2_model)
            print("[SAMWrapper] Predictor ready.")
        except Exception:
            raise

    def set_image(self, frame_bgr):
        self.predictor.set_image(frame_bgr)

    def predict_boxes(self, boxes):
        if self.predictor is None:
            return [], []
        boxes_tensor = np.array(boxes)
        masks_out, scores_out, _ = self.predictor.predict(
            point_coords=None,
            point_labels=None,
            box=boxes_tensor,
            multimask_output=False
        )
        masks_bool = []
        for i, mask_item in enumerate(masks_out):
            mask_arr = np.array(mask_item)
            if mask_arr.ndim == 3 and mask_arr.shape[0] == 1:
                mask_arr = mask_arr.squeeze(0)
            masks_bool.append(mask_arr > 0)
        return masks_bool, list(scores_out)


# ==============================================================================
# 7. MAIN VIDEO PROCESSING FUNCTION (DETECTION)
# ==============================================================================
def process_video_save_masks(input_path: str, out_json: str = OUT_JSON, out_overlay: str = OUT_OVERLAY, out_masks: str = OUT_MASKS):
    ensure_dir(out_masks)
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video {input_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    orig_W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    W = int(orig_W * RESIZE_SCALE)
    H = int(orig_H * RESIZE_SCALE)

    print("[INFO] estimating median background ...")
    bg_median = estimate_median_background(input_path, num_frames=BACKGROUND_MEDIAN_FRAMES, scale=RESIZE_SCALE)
    if bg_median is None:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        ret, bg_median = cap.read()
        if not ret:
            raise RuntimeError("Unable to read frames for background fallback.")
        if RESIZE_SCALE != 1.0:
            bg_median = cv2.resize(bg_median, (W,H))

    sam = SAMWrapper(CONFIG_FILE, CHECKPOINT, device)

    writer = None
    if out_overlay:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(out_overlay, fourcc, fps, (W, H))
        print(f"[INFO] writing overlay video -> {out_overlay}")

    results = {
        'video': os.path.basename(input_path),
        'fps': fps,
        'width': W,
        'height': H,
        'frames': []
    }

    frame_buffer = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    ret, f0 = cap.read()
    if f0 is not None and RESIZE_SCALE != 1.0: f0 = cv2.resize(f0, (W,H))

    ret, f1 = cap.read()
    if f1 is not None and RESIZE_SCALE != 1.0: f1 = cv2.resize(f1, (W,H))

    if f0 is not None: frame_buffer.append(f0)
    if f1 is not None: frame_buffer.append(f1)

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    prev_f = None
    curr_f = None
    next_f = None

    ret, frame = cap.read()
    if ret:
        if RESIZE_SCALE != 1.0: frame = cv2.resize(frame, (W,H))
        curr_f = frame

    ret, frame = cap.read()
    if ret:
        if RESIZE_SCALE != 1.0: frame = cv2.resize(frame, (W,H))
        next_f = frame

    processing_idx = 0
    t0 = time.time()

    while curr_f is not None:
        if processing_idx >= MAX_FRAMES_TO_PROCESS:
            print(f"[INFO] Reached limit of {MAX_FRAMES_TO_PROCESS} frames. Stopping.")
            break

        temp_next = next_f if next_f is not None else curr_f
        temp_prev = prev_f if prev_f is not None else curr_f

        boxes = generate_candidate_boxes(curr_f, temp_prev, temp_next, bg_median)
        frame_record = {'frame_idx': processing_idx, 'detections': []}

        if len(boxes) == 0:
            results['frames'].append(frame_record)
            if writer: writer.write(curr_f)
        else:
            sam.set_image(curr_f)
            masks_bool_list, scores = sam.predict_boxes(boxes)
            vis = curr_f.copy() if writer else None

            for det_i, box in enumerate(boxes):
                x1,y1,x2,y2 = box
                mask_bool = masks_bool_list[det_i]

                if mask_bool.shape != (H, W):
                    w_box = max(1, x2 - x1); h_box = max(1, y2 - y1)
                    try:
                        mask_resized = cv2.resize((mask_bool>0).astype('uint8'), (w_box, h_box), interpolation=cv2.INTER_NEAREST).astype(bool)
                    except:
                        mask_resized = np.zeros((h_box, w_box), dtype=bool)
                    full_mask = np.zeros((H, W), dtype=bool)
                    x2c = min(x1 + w_box, W)
                    y2c = min(y1 + h_box, H)
                    full_mask[y1:y2c, x1:x2c] = mask_resized[:(y2c-y1), :(x2c-x1)]
                else:
                    full_mask = mask_bool.astype(bool)

                area = int(full_mask.sum())
                if area < MIN_AREA or area > MAX_AREA:
                    continue

                mask_filename = f"mask_f{processing_idx:06d}_d{det_i:03d}.png"
                mask_path = os.path.join(out_masks, mask_filename)
                cv2.imwrite(mask_path, (full_mask.astype('uint8')*255))

                poly = mask_to_polygon(full_mask)
                bbox = mask_to_bbox(full_mask)

                mask_u8 = (full_mask.astype('uint8')*255)
                cts, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if cts:
                    cnt = max(cts, key=cv2.contourArea)
                    perimeter = float(cv2.arcLength(cnt, True))
                    hull = cv2.convexHull(cnt)
                    hull_area = float(cv2.contourArea(hull)) if hull is not None else 0.0
                    solidity = float(area / hull_area) if hull_area > 1e-6 else 0.0
                else:
                    perimeter = 0.0; hull_area = 0.0; solidity = 0.0

                centroid = [float(np.mean(np.where(full_mask)[1])), float(np.mean(np.where(full_mask)[0]))] if area>0 else [0.0,0.0]

                det_entry = {
                    'box_idx': det_i,
                    'box': [int(x1), int(y1), int(x2), int(y2)],
                    'area': area,
                    'centroid': centroid,
                    'bbox_xywh': bbox,
                    'polygon': poly,
                    'perimeter': perimeter,
                    'hull_area': hull_area,
                    'solidity': solidity,
                    'mask_file': mask_path,
                    'score': float(scores[det_i]) if (scores is not None and det_i < len(scores)) else None
                }
                frame_record['detections'].append(det_entry)

                if writer:
                    m_bool = (full_mask > 0)
                    color = (0,200,0)
                    vis[m_bool] = (vis[m_bool] * 0.4 + np.array(color) * 0.6).astype('uint8')
                    if poly:
                        pts = np.array(poly, dtype=np.int32).reshape(-1,1,2)
                        cv2.polylines(vis, [pts], True, (0,0,0), 1)
                    cv2.rectangle(vis, (x1,y1), (x2,y2), (0,128,255), 1)

            if writer:
                writer.write(vis)

        results['frames'].append(frame_record)

        if processing_idx % 20 == 0:
            elapsed = time.time() - t0
            avg_ms = elapsed / (processing_idx+1) * 1000.0
            print(f"[INFO] frame {processing_idx}/{total_frames} processed | detections: {len(frame_record['detections'])} | avg {avg_ms:.1f} ms/frame")

        prev_f = curr_f
        curr_f = next_f

        ret, frame = cap.read()
        if ret:
            if RESIZE_SCALE != 1.0: frame = cv2.resize(frame, (W,H))
            next_f = frame
        else:
            next_f = None

        processing_idx += 1

    cap.release()
    if writer:
        writer.release()

    with open(out_json, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"[INFO] wrote JSON -> {out_json}")
    print(f"[INFO] masks saved in directory -> {out_masks}")


In [12]:
# ==============================================================================
# 8. TRACKER UTILITIES & DATA STRUCTURES
# ==============================================================================
def resolve_mask_path(path, base_dir):
    if os.path.isabs(path) and os.path.exists(path):
        return path

    p1 = os.path.join(base_dir, path)
    if os.path.exists(p1):
        return p1

    p2 = os.path.join(base_dir, TRACKER_CONFIG["input_masks"], os.path.basename(path))
    if os.path.exists(p2):
        return p2

    p3 = os.path.join(TRACKER_CONFIG["input_masks"], os.path.basename(path))
    if os.path.exists(p3):
        return p3

    return None

def load_mask(path, base_dir, target_shape):
    full = resolve_mask_path(path, base_dir)
    if full is None:
        return None

    mask = cv2.imread(full, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None

    mask = (mask > 127).astype(np.uint8)

    if mask.shape[0] != target_shape[0] or mask.shape[1] != target_shape[1]:
        mask = cv2.resize(
            mask,
            (target_shape[1], target_shape[0]),
            interpolation=cv2.INTER_NEAREST
        )
    return mask

def mask_iou(m1, m2):
    if m1 is None or m2 is None:
        return 0.0
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    return 0.0 if union == 0 else inter / union

def centroid_from_mask(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return (0.0, 0.0)
    return (float(xs.mean()), float(ys.mean()))

def color_from_id(tid):
    np.random.seed(tid)
    return tuple(int(x) for x in np.random.randint(64, 255, 3))


class Track:
    def __init__(self, track_id, det, frame_idx):
        self.id = track_id
        self.last_mask = det["mask"]
        self.last_centroid = det["centroid"]
        self.missed = 0
        self.age = 1
        self.confirmed = False
        self.history = []

        self.history.append({
            "frame": frame_idx,
            "centroid": list(det["centroid"]),
            "mask_file": det["mask_file"]
        })

    def update(self, det, frame_idx):
        self.last_mask = det["mask"]
        self.last_centroid = det["centroid"]
        self.missed = 0
        self.age += 1
        if self.age >= TRACKER_CONFIG["min_confirm_frames"]:
            self.confirmed = True

        self.history.append({
            "frame": frame_idx,
            "centroid": list(det["centroid"]),
            "mask_file": det["mask_file"]
        })

    def mark_missed(self):
        self.missed += 1

    def dead(self):
        return self.missed > TRACKER_CONFIG["max_missed_frames"]


# ==============================================================================
# 9. TRACKER CORE PIPELINE FUNCTIONS
# ==============================================================================
def parse_frame_detections(frame_rec, base_dir, target_shape):
    detections = []
    if frame_rec:
        for d in frame_rec.get("detections", []):
            mask = load_mask(d["mask_file"], base_dir, target_shape)
            if mask is None:
                continue
            detections.append({
                "mask": mask,
                "centroid": centroid_from_mask(mask),
                "mask_file": d["mask_file"]
            })
    return detections

def match_detections_to_tracks(tracks, detections, claimed, frame_idx):
    for t in tracks:
        best_iou = 0.0
        best_idx = None

        for i, det in enumerate(detections):
            if i in claimed:
                continue

            dx = t.last_centroid[0] - det["centroid"][0]
            dy = t.last_centroid[1] - det["centroid"][1]
            if dx*dx + dy*dy > TRACKER_CONFIG["max_centroid_dist"]**2:
                continue

            iou = mask_iou(t.last_mask, det["mask"])
            if iou > best_iou:
                best_iou = iou
                best_idx = i

        if best_idx is not None and best_iou >= TRACKER_CONFIG["iou_match_thresh"]:
            t.update(detections[best_idx], frame_idx)
            claimed.add(best_idx)
        else:
            t.mark_missed()

def spawn_new_tracks(tracks, detections, claimed, frame_idx, next_id):
    for i, det in enumerate(detections):
        if i in claimed:
            continue

        spawn = True
        for t in tracks:
            if mask_iou(t.last_mask, det["mask"]) > TRACKER_CONFIG["iou_spawn_thresh"]:
                spawn = False
                break

        if spawn:
            tracks.append(Track(next_id, det, frame_idx))
            next_id += 1

    return next_id

def visualize_frame(frame, tracks, frame_idx):
    vis = frame.copy()

    for t in tracks:
        if not t.confirmed:
            continue

        color = color_from_id(t.id)
        mask = t.last_mask

        overlay = np.zeros_like(vis)
        overlay[mask == 1] = color
        vis = cv2.addWeighted(vis, 1.0, overlay, TRACKER_CONFIG["mask_alpha"], 0)

        cx, cy = map(int, t.last_centroid)
        cv2.putText(vis, str(t.id), (cx, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        if TRACKER_CONFIG["draw_trails"]:
            pts = [(int(h["centroid"][0]), int(h["centroid"][1]))
                   for h in t.history[-TRACKER_CONFIG["trail_length"]:]]
            for j in range(1, len(pts)):
                cv2.line(vis, pts[j-1], pts[j], color, 2)

    cv2.putText(vis,
                f"Frame {frame_idx} | Active Tracks {len(tracks)}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255, 255, 255),
                2)
    return vis

def save_tracks_to_json(tracks, video_name):
    output_tracks = []
    for t in tracks:
        if t.confirmed:
            output_tracks.append({
                "track_id": t.id,
                "start_frame": t.history[0]["frame"],
                "end_frame": t.history[-1]["frame"],
                "history": t.history
            })

    with open(TRACKER_CONFIG["output_json"], "w") as f:
        json.dump({
            "video": video_name,
            "num_tracks": len(output_tracks),
            "tracks": output_tracks
        }, f, indent=2)

    return output_tracks


# ==============================================================================
# 10. MAIN MASK-IOU TRACKER
# ==============================================================================
def run_mask_iou():
    print("\n--- Starting Mask-IoU Tracker ---")
    print("Loading detections...")
    with open(TRACKER_CONFIG["input_json"], "r") as f:
        data = json.load(f)

    base_dir = os.path.dirname(TRACKER_CONFIG["input_json"]) or os.getcwd()

    print("Opening video...")
    cap = cv2.VideoCapture(TRACKER_CONFIG["input_video"])
    if not cap.isOpened():
        raise RuntimeError("Could not open input video")

    fps = cap.get(cv2.CAP_PROP_FPS)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(
        TRACKER_CONFIG["output_video"],
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (W, H)
    )

    frames_dict = {f["frame_idx"]: f for f in data["frames"]}

    # Handle empty detection files
    if not frames_dict:
        print("No detections found. Exiting tracker.")
        cap.release()
        writer.release()
        return

    max_frame = max(frames_dict.keys())

    tracks = []
    next_id = 1

    # --- MAIN PROCESSING LOOP ---
    for frame_idx in range(max_frame + 1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            continue

        # 1. Load Detections
        frame_rec = frames_dict.get(frame_idx)
        detections = parse_frame_detections(frame_rec, base_dir, (H, W))
        claimed = set()

        # 2. Match & Update Existing Tracks
        match_detections_to_tracks(tracks, detections, claimed, frame_idx)

        # 3. Spawn New Tracks
        next_id = spawn_new_tracks(tracks, detections, claimed, frame_idx, next_id)

        # 4. Cleanup Dead Tracks
        tracks = [t for t in tracks if not t.dead()]

        # 5. Visualize & Save Frame
        vis = visualize_frame(frame, tracks, frame_idx)
        writer.write(vis)

        if frame_idx % 10 == 0:
            print(f"Processed {frame_idx}/{max_frame}", end="\r")

    cap.release()
    writer.release()

    # --- SAVE RESULTS ---
    video_name = data.get("video", "unknown")
    output_tracks = save_tracks_to_json(tracks, video_name)

    print("\nDone.")
    print("Tracks:", len(output_tracks))
    print("Video:", TRACKER_CONFIG["output_video"])


In [ ]:
# ==============================================================================
# 11. CLI & END-TO-END EXECUTION
# ==============================================================================
def main():
    parser = argparse.ArgumentParser(description="End-to-End SAM 2 Detection and Mask-IoU Tracking Pipeline")

    # --- Detection Arguments (Defaults to None so we can handle fallbacks manually) ---
    parser.add_argument('--input', '-i', type=str, default=None, help='Input video file path')
    parser.add_argument('--out_json', type=str, default=None, help='Output JSON file for SAM 2 detections')
    parser.add_argument('--out_overlay', type=str, default=None, help='Optional overlay video for detection phase')
    parser.add_argument('--out_masks', type=str, default=None, help='Output masks directory')

    # --- Tracking Arguments ---
    parser.add_argument('--in_json', type=str, default=None, help='Input JSON for tracking')
    parser.add_argument('--track_out_json', type=str, default=None, help='Final output JSON for tracked data')
    parser.add_argument('--track_out_video', type=str, default=None, help='Final output video with tracking trails')
    parser.add_argument('--track_in_masks', type=str, default=None, help='Input- detected masks directory')

    # --- Execution Flags ---
    parser.add_argument('--detect_only', action='store_true', help='Run ONLY the SAM 2 detection phase')
    parser.add_argument('--track_only', action='store_true', help='Run ONLY the Mask-IoU tracking phase')

    args = parser.parse_args()

    if args.detect_only and args.track_only:
        print("[ERROR] You cannot use both --detect_only and --track_only together.")
        return

    # ==========================================
    # 1. RUN DETECT ONLY
    # ==========================================
    if args.detect_only:
        print("\n=== MODE 1: SAM 2 DETECTION ONLY ===")
        # Use CLI if provided, else use global config defaults
        det_input = args.input if args.input is not None else IN_VIDEO
        det_out_json = args.out_json if args.out_json is not None else OUT_JSON
        det_overlay = args.out_overlay if args.out_overlay is not None else OUT_OVERLAY
        det_masks = args.out_masks if args.out_masks is not None else OUT_MASKS

        process_video_save_masks(det_input, det_out_json, det_overlay, det_masks)

    # ==========================================
    # 2. RUN TRACK ONLY
    # ==========================================
    elif args.track_only:
        print("\n=== MODE 2: MASK-IOU TRACKING ONLY ===")
        # Use CLI if provided, else strictly use whatever is in TRACKER_CONFIG
        TRACKER_CONFIG["input_video"] = args.input if args.input is not None else TRACKER_CONFIG["input_video"]
        TRACKER_CONFIG["input_json"] = args.in_json if args.in_json is not None else TRACKER_CONFIG["input_json"]
        TRACKER_CONFIG["input_masks"] = args.in_masks if args.track_in_masks is not None else TRACKER_CONFIG["input_masks"]
        TRACKER_CONFIG["output_json"] = args.track_out_json if args.track_out_json is not None else TRACKER_CONFIG["output_json"]
        TRACKER_CONFIG["output_video"] = args.track_out_video if args.track_out_video is not None else TRACKER_CONFIG["output_video"]

        run_mask_iou()

    # ==========================================
    # 3. RUN FULL PIPELINE
    # ==========================================
    else:
        print("\n=== MODE 3: FULL PIPELINE (DETECT -> TRACK) ===")

        # --- Phase 1: Detect ---
        # Uses CLI if provided, else uses config defaults
        det_input = args.input if args.input is not None else IN_VIDEO
        det_out_json = args.out_json if args.out_json is not None else OUT_JSON
        det_overlay = args.out_overlay if args.out_overlay is not None else OUT_OVERLAY
        det_masks = args.out_masks if args.out_masks is not None else OUT_MASKS
        print("\n--- Running Detection ---")
        process_video_save_masks(det_input, det_out_json, det_overlay, det_masks)

        # --- Phase 2: Track ---
        print("\n--- Running Tracking ---")
        # STRICT RULE: Tracking inputs MUST be the outputs of Phase 1
        TRACKER_CONFIG["input_video"] = det_overlay
        TRACKER_CONFIG["input_json"] = det_out_json
        TRACKER_CONFIG["input_masks"] = det_masks
        # Tracking outputs use CLI if provided, else fallback to TRACKER_CONFIG defaults
        TRACKER_CONFIG["output_json"] = args.track_out_json if args.track_out_json is not None else TRACKER_CONFIG["output_json"]
        TRACKER_CONFIG["output_video"] = args.track_out_video if args.track_out_video is not None else TRACKER_CONFIG["output_video"]

        run_mask_iou()

if __name__ == '__main__':
    main()